# M8 · 개인 대시보드와 RAG 통합

개인용 `app_personal.py`는 ②~⑤ 집계 현황과 출처 있는 질문 기능을 같은 화면에 배치한다. ① 사례 분포도 별도로 보여준다. 먼저 저장 데이터·색인이 같은 버전인지 확인하고, 앱을 자동 실행해 필터와 화면 오류를 검사한다.

In [ ]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
P=ROOT/'data'/'personal'
dashboard=pd.read_parquet(P/'processed'/'dashboard_metrics.parquet')
sif=pd.read_parquet(P/'processed'/'source_01_sif_cases.parquet')
lexical=json.loads((P/'corpus'/'index_meta.json').read_text(encoding='utf-8'))
semantic=json.loads((P/'corpus'/'index_meta_semantic.json').read_text(encoding='utf-8'))
assert len(dashboard)==300 and len(sif)==6032
assert semantic['complete'] and semantic['chunks']==lexical['chunks']==6845
assert semantic['corpus_sha256']==lexical['corpus_sha256']==hashlib.sha256((P/'corpus'/'chunks.jsonl').read_bytes()).hexdigest()
display({'대시보드 칸':len(dashboard),'SIF 사례':len(sif),'RAG 청크':semantic['chunks'],'의미 색인 완료':semantic['complete']})

## Streamlit 화면과 업종 필터 검사

In [ ]:
from streamlit.testing.v1 import AppTest
app=AppTest.from_file(str(ROOT/'app_personal.py')).run(timeout=60)
assert not app.exception, [str(x.message) for x in app.exception]
assert len(app.metric)==4
print('기본 화면 예외:',len(app.exception),'집계 카드:',len(app.metric))
before=app.metric[0].value
app.sidebar.selectbox[0].set_value('건설업').run(timeout=60)
assert not app.exception
after=app.metric[0].value
assert before!=after
print('건설업 선택 전후 사고재해자:',before,'→',after)

## 브라우저 질문 시험 기록

In [ ]:
spot=json.loads((P/'evaluation'/'answer_spotcheck_multisource.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'질문':r['id'],'상태':r['status'],'인용 수':len(r['sources'])} for r in spot]))
assert len(spot)==8
tbm=next(r for r in spot if r['id']=='TBM')
assert tbm['status']=='answered'
assert {'sif','kosha_guide'} <= {s['source_type'] for s in tbm['sources']}
print('TBM 출처 유형:',sorted({s['source_type'] for s in tbm['sources']}))
assert all(r['status']=='insufficient_evidence' for r in spot if r['id'].startswith('NEG'))
print('앱 실행: uv run streamlit run app_personal.py --server.port 8502')

## 앱 통합 확인 결과

- 앱이 읽는 대시보드 300칸, SIF 6,032건, 검색 코퍼스 청크 6,845개의 수와 코퍼스 SHA-256을 확인했다. semantic Chroma 색인도 6,845개이며 코퍼스 해시가 일치했다.
- Streamlit AppTest에서 화면 예외 없이 지표 카드 4개가 만들어지고, 업종을 전체에서 건설업으로 바꾸면 사고재해자 합계가 달라지는 것을 확인했다. 대시보드와 RAG 질문이 한 화면에 배치되어 있다.
- 저장된 답변 표본 8개에서는 긍정 질문과 무관 질문의 모름 처리를 확인했다. TBM 예시는 SIF와 KOSHA GUIDE 양쪽 자료 유형의 근거를 인용했다. 이 표본은 생성 답변의 전체 정확도를 증명하지 않는다.
- 자동 화면검사와 기존 저장 표본은 브라우저에서 새로 LLM 질문을 전송한 실시간 종단 테스트와 다르다. 실제 질문 입력·응답 후 화면 근거 링크를 여는 과정은 사용자가 본 브라우저에서 별도 최종 시연 점검이 필요하다.
- OCR 결과는 PDF 원문 대조가 필요하고, 미적재 6쪽은 검색 누락 가능성이 있다. 답변의 인용 URL·페이지·자료 유형을 확인하고, 근거가 약하면 답을 채택하지 않는다.

### 사용 및 공유 전에 남은 것

- 2026-10-03 M0·M1을 다시 실행했다. ②~⑥ API가 모두 정상 응답했고, ②~⑤ 각 30행과 ⑥ 5행의 전체 응답이 원본 CSV와 일치했다.
- SIF 최신 게시본은 로컬 XLSX와 SHA-256이 일치했다. ①·⑥의 출처표시·변경금지 범위와 팀 내 산출물 공유 가능 여부는 여전히 확정되지 않았다.
- 현재 앱은 개인 작업용 `app_personal.py`다. 프로젝트 공개·배포, 데이터 재배포, 안전 관련 현장 의사결정의 근거로 쓰기 전에는 이용조건·운영 보안·답변 품질 검토가 별도로 필요하다.
